# 24 — Experimento para mejorar el pronóstico semanal

## Pregunta y alcance

¿Podemos conservar el desempeño de H1 y reducir el error de H2–H5 frente al ingeniero? Este cuaderno compara, en la misma población y con la misma política de entrenamiento: (1) el modelo agrupado actual, (2) un modelo de boosting por horizonte y (3) el modelo por horizonte con una corrección de sesgo estimada en un periodo anterior.

El n−1 no se usa como solución general: sólo retirar clavel ayudó en un corte y retirar miniclavel o ambos empeoró. En este cuaderno se revisa dónde se concentran los errores por producto y color, sin excluirlos del entrenamiento.

## Diseño temporal

1. Tuning con orígenes de noviembre de 2025.
2. Factor de corrección por horizonte calculado en orígenes de diciembre de 2025.
3. Evaluación final en las semanas completas recientes de 2026 comparables con el ingeniero.

En cada origen, el entrenamiento sólo ve semanas objetivo cerradas antes del origen. Para que los ajustes corran con la memoria disponible, cada entrenamiento usa una muestra reproducible de hasta 5.000 filas. La comparación principal es dentro de este cuaderno, donde agrupado y separado usan la misma regla. Los resultados recientes ya revisados no se reutilizan para elegir hiperparámetros ni el factor de sesgo.

**Lectura de métricas:** WAPE más bajo significa menos error absoluto relativo; sesgo más cercano a cero significa menos sobre o subestimación neta. El ingeniero es el benchmark, no una variable de entrada.

## 1. Cargar datos y construir la población comparable

In [1]:
# Datos comunes y evaluación sin fuga temporal
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from threadpoolctl import threadpool_limits

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Datos_analiticos_proyecto").is_dir())
DATA = ROOT / "Datos_analiticos_proyecto"
OUT = DATA / "experimentos_actualizacion_20261020"
OUT.mkdir(parents=True, exist_ok=True)
DEST = DATA / "modelo_con_proyecciones_teoricas"
K = ["finca", "producto", "color", "variedad"]
CASE = K + ["origen", "semana_objetivo", "horizonte"]

# Features calculadas al origen. y se sustituye por la producción 2026 recién limpia,
# pero las entradas históricas del caso quedan congeladas como estaban al emitir.
base = pd.read_parquet(DEST / "base_analitica_proyecciones.parquet")
real26 = pd.read_parquet(DATA / "produccion_real_2026_actualizada.parquet")
real26 = real26.loc[real26.finca.eq("GAITANA") & real26.llave_completa].copy()
real_semanal = real26.groupby(K + ["semana_inicio"], as_index=False).agg(
    y_nuevo=("tallos", "sum"), ultimo_dia_real=("fecha_corte", "max"))
ultima_fecha = real26.fecha_corte.max()
ultima_semana = ultima_fecha - pd.Timedelta(days=ultima_fecha.weekday())
ultima_semana_completa = ultima_semana if ultima_fecha.weekday() == 6 else ultima_semana - pd.Timedelta(days=7)

casos = base.loc[base.finca.eq("GAITANA")].copy()
casos = casos.merge(real_semanal, left_on=K + ["semana_objetivo"],
                    right_on=K + ["semana_inicio"], how="left", validate="many_to_one")
casos["y_eval"] = casos.y_nuevo.where(casos.y_nuevo.notna(), casos.y)
casos["admisible_actual"] = casos.media_4.notna() & casos.observaciones_previas.ge(26)
casos["horizonte"] = casos.horizonte.astype(int)
model_data = casos.loc[casos.y_eval.notna() & casos.admisible_actual].copy()

# Mismo cruce nominal estricto del benchmark oficial: suma las líneas de la misma
# serie/origen/objetivo/horizonte y excluye cantidades incompletas y envíos posteriores.
e = pd.read_parquet(DATA / "estimados_semanales_limpio.parquet")
e = e.loc[e.finca.eq("GAITANA") & e.en_h1_h5 & ~e.es_ajustado
          & e.origen_solicitado.dt.year.eq(2026) & e.semana_inicio.dt.year.eq(2026)].copy()
e = e.rename(columns={"origen_solicitado": "origen", "semana_inicio": "semana_objetivo"})
engineer = e.groupby(CASE, dropna=False).agg(
    ingeniero=("estimado", "sum"), lineas=("estimado", "size"),
    cantidades=("estimado", "count"), archivos=("archivo_origen", "nunique"),
    posterior=("archivo_es_posterior", "max")).reset_index()
engineer["ingeniero_completo"] = engineer.lineas.eq(engineer.cantidades) & engineer.archivos.eq(1) & ~engineer.posterior.fillna(False)
casos = casos.merge(engineer, on=CASE, how="inner", validate="one_to_one")
casos = casos.loc[casos.ingeniero_completo & casos.y_eval.notna() & casos.admisible_actual].copy()
casos = casos.loc[casos.semana_objetivo.gt(pd.Timestamp("2026-08-31"))
                  & casos.semana_objetivo.le(ultima_semana_completa)].copy()
print("Última fecha real:", ultima_fecha.date(), "| última semana completa:", ultima_semana_completa.date())
print("Casos nuevos comparables tras cruce estricto:", len(casos))
print("Semanas objetivo:", sorted(casos.semana_objetivo.dt.strftime("%Y-%m-%d").unique()))
print("Clima disponible hasta:", pd.to_datetime(pd.read_parquet(DATA / "clima_limpio.parquet").fecha).max().date())


Última fecha real: 2026-10-03 | última semana completa: 2026-09-21
Casos nuevos comparables tras cruce estricto: 1226
Semanas objetivo: ['2026-09-07', '2026-09-14', '2026-09-21']
Clima disponible hasta: 2026-09-21


## 2. Métricas y reglas de entrenamiento

In [2]:
CATS = ["producto", "color", "variedad"]
MEMORY = ["produccion_lag1", "produccion_lag2", "produccion_lag3", "media_4", "media_8",
          "desv_4", "tendencia_4_8", "cambio_reciente", "produccion_52"]
CLIMATE = ["temperatura_semana_lag1", "temperatura_semana_media4",
           "humedad_semana_lag1", "humedad_semana_media4",
           "radiacion_semana_lag1", "radiacion_semana_media4"]
THEORY = ["teoria_modelo", "antiguedad_curva", "desviacion_reciente_teoria",
          "teoria_previa_1", "teoria_previa_2", "revision_teoria"]
BASE_X = CATS + ["horizonte", "semana_seno", "semana_coseno"] + MEMORY

def metricas(g, p):
    y = np.asarray(g.y_eval, dtype=float); p = np.asarray(p, dtype=float)
    den = y.sum()
    return {"n": len(y), "MAE_tallos": float(np.mean(np.abs(y-p))),
            "WAPE_pct": float(100*np.abs(y-p).sum()/den) if den else np.nan,
            "sesgo_pct": float(100*(p-y).sum()/den) if den else np.nan}

def elegir_clima(train, test, habilitar=True, minimo=0.80):
    """Usa clima sólo si existe cobertura suficiente en el ajuste evaluado."""
    if not habilitar or not all(c in train and c in test for c in CLIMATE):
        return [], 0.0, "sin clima: columnas no disponibles"
    cov_train = float(train[CLIMATE].notna().mean().mean())
    latest_origin = test.origen.max()
    current = test.loc[test.origen.eq(latest_origin), CLIMATE]
    cov_current = float(current.notna().mean().mean()) if len(current) else 0.0
    cobertura = min(cov_train, cov_current)
    if cobertura < minimo:
        return [], cobertura, f"sin clima: cobertura {cobertura:.1%} < {minimo:.0%}"
    return CLIMATE, cobertura, f"con clima: cobertura {cobertura:.1%}"

def crear_modelo(cols, params=None):
    params = params or {}
    cats = [c for c in CATS if c in cols]
    nums = [c for c in cols if c not in cats]
    prep = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32), cats),
        ("num", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True), nums),
    ])
    reg = HistGradientBoostingRegressor(
        learning_rate=params.get("learning_rate", .08), max_iter=params.get("max_iter", 80),
        max_leaf_nodes=params.get("max_leaf_nodes", 15), min_samples_leaf=40,
        l2_regularization=1.0, early_stopping=False, random_state=42)
    return make_pipeline(prep, reg)

def pronosticar_ruta(train, test, cols, params=None):
    """Ajusta la media4 y, con soporte suficiente, la teoría completa por separado."""
    train = train.loc[train.y_eval.notna() & train.media_4.notna()].copy()
    pred = pd.Series(index=test.index, dtype=float)
    if train.empty or test.empty: return pred
    if len(train) > 5000: train = train.sample(5000, random_state=42)
    with threadpool_limits(limits=2):
        fallback = crear_modelo(cols, params)
        fallback.fit(train[cols], train.y_eval-train.media_4)
        pred.loc[test.index] = np.maximum(0, test.media_4 + fallback.predict(test[cols]))
        tr = train.loc[train.curva_completa.fillna(False) & train.teoria_modelo.notna()]
        te = test.loc[test.curva_completa.fillna(False) & test.teoria_modelo.notna()]
        if len(tr) >= 40 and len(te):
            use = [c for c in cols + THEORY if c in train.columns]
            teoria = crear_modelo(use, params)
            teoria.fit(tr[use], tr.y_eval-tr.teoria_modelo)
            pred.loc[te.index] = np.maximum(0, te.teoria_modelo + teoria.predict(te[use]))
    return pred

def fuera_producto(df, patron):
    return df.producto.isin(str(patron).split("|"))

def evaluar_origenes(data, grupos="agrupado", excluir_entrenamiento=None, clima="auto", params=None, train_pool=None):
    """Reentrena sólo con semanas reales cerradas antes de cada origen evaluado."""
    train_pool = model_data if train_pool is None else train_pool
    salidas=[]; controles=[]
    for origen, test in data.groupby("origen", sort=True):
        test = test.copy()
        train = train_pool.loc[train_pool.semana_objetivo + pd.Timedelta(days=7) <= origen].copy()
        if excluir_entrenamiento:
            train = train.loc[~fuera_producto(train, excluir_entrenamiento)]
        if train.empty: continue
        climat, cov, modo = elegir_clima(train, test, habilitar=(clima != "sin"))
        x = BASE_X + climat
        test["prediccion"] = np.nan
        grp = test.horizonte if grupos == "horizonte" else np.where(test.horizonte <= 2, "H1_H2", "H3_H5")
        test["_grp"] = grp
        train["_grp"] = train.horizonte if grupos == "horizonte" else np.where(train.horizonte <= 2, "H1_H2", "H3_H5")
        for gval, te in test.groupby("_grp"):
            tr = train.loc[train._grp.eq(gval)]
            if tr.empty: continue
            teorio = x + THEORY
            te["prediccion"] = pronosticar_ruta(tr, te, x, params)
            test.loc[te.index, "prediccion"] = te.prediccion
            if test.loc[te.index, "prediccion"].notna().any():
                controles.append({"origen": origen, "grupo": str(gval), "train_n": len(tr),
                                  "test_n": len(te), "clima": modo, "cobertura_clima": cov})
        test.drop(columns="_grp", inplace=True)
        salidas.append(test)
    if not salidas: return pd.DataFrame(), pd.DataFrame(controles)
    return pd.concat(salidas).dropna(subset=["prediccion"]), pd.DataFrame(controles)

def resumen_comparacion(pred):
    rows=[]
    for nombre,col in [("Modelo", "prediccion"),("Ingeniero","ingeniero"),("Media 4 semanas","media_4"),("Teoría aplicable","teoria_modelo")]:
        g=pred.loc[pred[col].notna()].copy()
        if len(g): rows.append({"metodo":nombre,**metricas(g,g[col])})
    return pd.DataFrame(rows)


## 3. Elegir parámetros con noviembre y congelarlos

In [3]:
# La misma cuadrícula corta se evalúa para el agrupado y para cada horizonte.
# No se mira el periodo final de 2026 para seleccionar parámetros.
valid_nov = model_data.loc[
    model_data.origen.between("2025-11-03", "2025-11-24")
    & model_data.y_eval.notna()
].copy()
grid = [
    {"max_iter": 40, "max_leaf_nodes": 15},
    {"max_iter": 80, "max_leaf_nodes": 15},
]
tuning_rows = []

for params in grid:
    pred_g, _ = evaluar_origenes(
        valid_nov, grupos="agrupado", clima="auto", params=params, train_pool=model_data
    )
    if len(pred_g):
        tuning_rows.append({"familia": "Agrupado H1-H2/H3-H5", **params,
                            **metricas(pred_g, pred_g.prediccion)})

for h in range(1, 6):
    vh = valid_nov.loc[valid_nov.horizonte.eq(h)].copy()
    for params in grid:
        pred_h, _ = evaluar_origenes(
            vh, grupos="horizonte", clima="auto", params=params, train_pool=model_data
        )
        if len(pred_h):
            tuning_rows.append({"familia": f"Modelo H{h}", **params,
                                **metricas(pred_h, pred_h.prediccion)})

tuning = pd.DataFrame(tuning_rows)
if tuning.empty:
    raise ValueError("No se generaron predicciones válidas para el tuning temporal de noviembre.")
best_params = (tuning.sort_values(["familia", "WAPE_pct", "max_iter"])
               .groupby("familia", as_index=False).first())
display(tuning.sort_values(["familia", "WAPE_pct"]))
display(best_params)


C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\entorno_tesis\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] El sistema no puede encontrar el archivo especificado
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\entorno_tesis\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
  File "C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\entorno_tesis\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".spli

,familia,max_iter,max_leaf_nodes,n,MAE_tallos,WAPE_pct,sesgo_pct
1,Agrupado H1-H2/H3-H5,80,15,2381,2189.444091,22.615321,0.527525
0,Agrupado H1-H2/H3-H5,40,15,2381,2235.646339,23.092555,1.461376
3,Modelo H1,80,15,485,1480.428815,14.638761,1.294373
2,Modelo H1,40,15,485,1531.141120,15.140214,0.537571
5,Modelo H2,80,15,486,2009.710890,19.946562,0.511471
4,Modelo H2,40,15,486,2033.181194,20.179506,-0.302098
6,Modelo H3,40,15,479,2502.581397,25.936707,1.358795
7,Modelo H3,80,15,479,2502.701915,25.937956,1.027290
8,Modelo H4,40,15,468,2654.140683,28.288886,3.781215
9,Modelo H4,80,15,468,2673.101496,28.490978,3.628313


,familia,max_iter,max_leaf_nodes,n,MAE_tallos,WAPE_pct,sesgo_pct
0,Agrupado H1-H2/H3-H5,80,15,2381,2189.444091,22.615321,0.527525
1,Modelo H1,80,15,485,1480.428815,14.638761,1.294373
2,Modelo H2,80,15,486,2009.710890,19.946562,0.511471
3,Modelo H3,40,15,479,2502.581397,25.936707,1.358795
4,Modelo H4,40,15,468,2654.140683,28.288886,3.781215
5,Modelo H5,80,15,463,2684.038410,29.331074,1.924991


## 4. Estimar la corrección de sesgo con diciembre

In [4]:
# Diciembre se usa sólo para estimar un factor multiplicativo por horizonte.
# La fórmula lleva la suma pronosticada hacia la suma real del periodo de calibración.
cal_dec = model_data.loc[
    model_data.origen.between("2025-12-01", "2025-12-28")
    & model_data.y_eval.notna()
].copy()
factors = {}
cal_rows = []

for h in range(1, 6):
    ph = best_params.loc[best_params.familia.eq(f"Modelo H{h}")]
    params = ph.iloc[0][["max_iter", "max_leaf_nodes"]].to_dict()
    dh = cal_dec.loc[cal_dec.horizonte.eq(h)].copy()
    pcal, _ = evaluar_origenes(
        dh, grupos="horizonte", clima="auto", params=params, train_pool=model_data
    )
    if pcal.empty or pcal.prediccion.sum() <= 0:
        factors[h] = 1.0
        cal_rows.append({"horizonte": h, "n_calibracion": len(pcal),
                         "factor": 1.0, "nota": "sin soporte; se deja sin ajuste"})
    else:
        factor = float(pcal.y_eval.sum() / pcal.prediccion.sum())
        factors[h] = factor
        cal_rows.append({"horizonte": h, "n_calibracion": len(pcal),
                         "WAPE_calibracion": metricas(pcal, pcal.prediccion)["WAPE_pct"],
                         "sesgo_calibracion": metricas(pcal, pcal.prediccion)["sesgo_pct"],
                         "factor": factor, "nota": "estimado antes del test final"})

calibracion = pd.DataFrame(cal_rows)
display(calibracion)


,horizonte,n_calibracion,WAPE_calibracion,sesgo_calibracion,factor,nota
0,1,470,16.038794,2.130445,0.979140,estimado antes del test final
1,2,468,22.166870,7.143291,0.933330,estimado antes del test final
2,3,468,27.297420,9.438606,0.913754,estimado antes del test final
3,4,467,27.983748,4.396396,0.957887,estimado antes del test final
4,5,467,33.116318,-5.634941,1.059714,estimado antes del test final


## 5. Comparar estrategias en las semanas nuevas

In [5]:
# Todos los métodos se puntúan en los mismos registros comparables con el ingeniero.
params_g = best_params.loc[best_params.familia.eq("Agrupado H1-H2/H3-H5")].iloc[0][
    ["max_iter", "max_leaf_nodes"]].to_dict()
pred_group, control_group = evaluar_origenes(
    casos, grupos="agrupado", clima="auto", params=params_g, train_pool=model_data
)

separate_parts = []
for h in range(1, 6):
    params_h = best_params.loc[best_params.familia.eq(f"Modelo H{h}")].iloc[0][
        ["max_iter", "max_leaf_nodes"]].to_dict()
    ph = casos.loc[casos.horizonte.eq(h)].copy()
    zh, _ = evaluar_origenes(
        ph, grupos="horizonte", clima="auto", params=params_h, train_pool=model_data
    )
    if len(zh):
        zh["factor_sesgo"] = factors[h]
        zh["pred_corregida"] = np.maximum(0, zh.prediccion * factors[h])
        separate_parts.append(zh)

pred_sep = pd.concat(separate_parts, ignore_index=True)
if pred_group.empty or pred_sep.empty:
    raise ValueError("Faltan predicciones para comparar estrategias en el mismo test.")

# Híbrido predefinido: conserva el grouped H1 y usa modelos separados en H2-H5.
# Se informa como experimento; la combinación se debe confirmar en cortes futuros.
keys = CASE
group_lookup = pred_group[keys + ["prediccion"]].rename(columns={"prediccion": "pred_group"})
hybrid = pred_sep.merge(group_lookup, on=keys, how="inner", validate="one_to_one")
hybrid["pred_hibrida"] = hybrid.prediccion.where(hybrid.horizonte.ne(1), hybrid.pred_group)
hybrid["pred_hibrida_cal"] = hybrid.pred_corregida.where(
    hybrid.horizonte.ne(1), hybrid.pred_group
)

strategies = [
    ("Agrupado ajustado en noviembre", pred_group, "prediccion"),
    ("Separado por horizonte", pred_sep, "prediccion"),
    ("Separado + corrección de sesgo diciembre", pred_sep, "pred_corregida"),
    ("H1 agrupado + H2-H5 separados", hybrid, "pred_hibrida"),
    ("H1 agrupado + H2-H5 separados corregidos", hybrid, "pred_hibrida_cal"),
]
summary_rows = []
for name, frame, col in strategies:
    summary_rows.append({"estrategia": name, **metricas(frame, frame[col])})
summary_rows.append({"estrategia": "Ingeniero", **metricas(pred_group, pred_group.ingeniero)})
summary = pd.DataFrame(summary_rows).sort_values("WAPE_pct")
display(summary)

# Detalle por horizonte, en la misma población del ingeniero.
detail_rows = []
for name, frame, col in strategies:
    for h, group in frame.groupby("horizonte"):
        detail_rows.append({"estrategia": name, "horizonte": int(h), **metricas(group, group[col])})
for h, group in pred_group.groupby("horizonte"):
    detail_rows.append({"estrategia": "Ingeniero", "horizonte": int(h),
                        **metricas(group, group.ingeniero)})
detail = pd.DataFrame(detail_rows)
display(detail)



,estrategia,n,MAE_tallos,WAPE_pct,sesgo_pct
5,Ingeniero,1226,2072.071778,16.343964,-7.507109
3,H1 agrupado + H2-H5 separados,1226,2257.184918,17.804089,-6.255499
1,Separado por horizonte,1226,2263.994173,17.857798,-6.174248
0,Agrupado ajustado en noviembre,1226,2314.541203,18.256500,-5.477878
2,Separado + corrección de sesgo diciembre,1226,2396.222427,18.900780,-9.004948
4,H1 agrupado + H2-H5 separados corregidos,1226,2399.320068,18.925214,-8.910191


,estrategia,horizonte,n,MAE_tallos,WAPE_pct,sesgo_pct
0,Agrupado ajustado en noviembre,1,103,1506.950594,12.278632,2.746177
1,Agrupado ajustado en noviembre,2,205,1840.841319,15.064884,1.969701
2,Agrupado ajustado en noviembre,3,306,2183.429658,17.023834,-9.012546
3,Agrupado ajustado en noviembre,4,306,2520.423088,19.651315,-6.679711
4,Agrupado ajustado en noviembre,5,306,2828.954880,22.056886,-8.143829
5,Separado por horizonte,1,103,1588.000569,12.939028,3.745207
6,Separado por horizonte,2,205,1827.538990,14.956021,-0.015055
7,Separado por horizonte,3,306,2151.275732,16.773135,-9.733610
8,Separado por horizonte,4,306,2497.363201,19.471521,-7.035950
9,Separado por horizonte,5,306,2663.280365,20.765149,-8.879382


## 6. Diagnóstico de errores por producto y color

In [6]:
# El diagnóstico usa el modelo agrupado y el ingeniero en exactamente las mismas filas.
eda = pred_group.copy()
eda["err_abs_modelo"] = (eda.prediccion - eda.y_eval).abs()
eda["err_abs_ingeniero"] = (eda.ingeniero - eda.y_eval).abs()
total_error_model = eda.err_abs_modelo.sum()
total_error_engineer = eda.err_abs_ingeniero.sum()

def tabla_aportes(df, group_cols):
    out = df.groupby(group_cols, dropna=False).agg(
        casos=("y_eval", "size"), real=("y_eval", "sum"),
        error_modelo=("err_abs_modelo", "sum"), error_ingeniero=("err_abs_ingeniero", "sum"),
        sesgo_modelo=("prediccion", lambda s: 100*(s.sum()-df.loc[s.index,"y_eval"].sum())/df.loc[s.index,"y_eval"].sum()),
        sesgo_ingeniero=("ingeniero", lambda s: 100*(s.sum()-df.loc[s.index,"y_eval"].sum())/df.loc[s.index,"y_eval"].sum()),
    ).reset_index()
    out["WAPE_modelo_pct"] = 100*out.error_modelo/out.real
    out["WAPE_ingeniero_pct"] = 100*out.error_ingeniero/out.real
    out["aporte_error_modelo_pct"] = 100*out.error_modelo/total_error_model
    out["aporte_error_ingeniero_pct"] = 100*out.error_ingeniero/total_error_engineer
    out["diferencia_error_abs"] = out.error_modelo-out.error_ingeniero
    return out.sort_values("diferencia_error_abs", ascending=False)

por_producto = tabla_aportes(eda, ["producto"])
por_color = tabla_aportes(eda, ["producto", "color"])
display(por_producto.head(10))
display(por_color.head(10))

OUT.mkdir(parents=True, exist_ok=True)
tuning.to_csv(OUT/"24_tuning_noviembre.csv", index=False)
calibracion.to_csv(OUT/"24_calibracion_sesgo_diciembre.csv", index=False)
summary.to_csv(OUT/"24_resumen_estrategias_semanales.csv", index=False)
detail.to_csv(OUT/"24_detalle_horizontes.csv", index=False)
por_producto.to_csv(OUT/"24_error_por_producto.csv", index=False)
por_color.to_csv(OUT/"24_error_por_color.csv", index=False)
pred_group.to_parquet(OUT/"24_predicciones_agrupado.parquet", index=False)
pred_sep.to_parquet(OUT/"24_predicciones_horizontes.parquet", index=False)
print("Tablas guardadas en", OUT)


,producto,casos,real,error_modelo,error_ingeniero,sesgo_modelo,sesgo_ingeniero,WAPE_modelo_pct,WAPE_ingeniero_pct,aporte_error_modelo_pct,aporte_error_ingeniero_pct,diferencia_error_abs
4,MINICARNATION,360,9131654,1395531.051151,1165718.0,-1.419509,-4.605124,15.282347,12.765683,49.179501,45.887906,229813.051151
2,CARNATION,461,3239324,832529.382946,704444.0,-19.646905,-8.130524,25.700714,21.746636,29.338924,27.730085,128085.382946
7,STAR,12,102835,43144.276597,16485.0,41.954856,12.840959,41.954856,16.030534,1.520435,0.648924,26659.276597
1,BARBATUS,60,95670,32208.964765,18800.0,5.122948,-17.455838,33.666734,19.650883,1.135067,0.740053,13408.964765
3,GREEN BALL,12,93960,16741.40967,12990.0,-12.840940,0.415070,17.817592,13.825032,0.589979,0.511345,3751.40967
9,VERONICA,24,29160,22673.768933,20160.0,77.756409,67.078189,77.756409,69.135802,0.79904,0.793588,2513.768933
0,AMAZON,12,20610,5161.794383,3670.0,23.098826,17.806890,25.045096,17.80689,0.181905,0.144468,1491.794383
10,VERONICA SPRAY,24,37530,11504.127351,12960.0,24.768142,34.532374,30.65315,34.532374,0.405414,0.510164,-1455.872649
8,STATICE,72,93600,30998.58385,35140.0,-11.068844,-30.235043,33.118145,37.542735,1.092412,1.383269,-4141.41615
5,RAFFINE,60,936036,183035.335783,227586.0,-6.556884,-19.643048,19.554305,24.313808,6.450295,8.958809,-44550.664217


,producto,color,casos,real,error_modelo,error_ingeniero,sesgo_modelo,sesgo_ingeniero,WAPE_modelo_pct,WAPE_ingeniero_pct,aporte_error_modelo_pct,aporte_error_ingeniero_pct,diferencia_error_abs
42,MINICARNATION,YELLOW,24,1025639,180911.934734,106357.0,-6.382790,8.001938,17.638948,10.369828,6.375464,4.18669,74554.934734
36,MINICARNATION,LIGHT PINK,36,852226,140060.594579,69188.0,9.264463,-6.644482,16.434677,8.118504,4.935834,2.723551,70872.594579
21,CARNATION,RED,29,286289,108120.827084,56079.0,-36.667925,-17.904635,37.766323,19.588248,3.810254,2.207522,52041.827084
37,MINICARNATION,ORANGE,36,1040075,200008.067535,148327.0,-15.168248,-10.548759,19.230158,14.261183,7.048426,5.838818,51681.067535
19,CARNATION,PEACH,36,322110,85367.031341,38212.0,-24.960807,4.883425,26.502447,11.863028,3.008395,1.504196,47155.031341
39,MINICARNATION,PURPLE,12,490642,73801.081493,31136.0,3.960168,-0.173650,15.041737,6.345971,2.600802,1.225653,42665.081493
18,CARNATION,ORANGE,36,327976,104765.26755,65818.0,-24.742640,1.257409,31.942968,20.067932,3.692002,2.590893,38947.26755
35,MINICARNATION,LAVENDER,12,533912,61000.907235,32012.0,-5.592116,-5.201606,11.425274,5.995745,2.149715,1.260136,28988.907235
34,MINICARNATION,HOT PINK,24,1049001,165838.664842,139025.0,1.162170,-4.657860,15.8092,13.253086,5.844272,5.47265,26813.664842
57,STAR,BICOLOR PINK,12,102835,43144.276597,16485.0,41.954856,12.840959,41.954856,16.030534,1.520435,0.648924,26659.276597


Tablas guardadas en C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\Datos_analiticos_proyecto\experimentos_actualizacion_20261020


## 7. Interpretación automática y decisión prudente

In [7]:
best = summary.sort_values("WAPE_pct").iloc[0]
best_bias = summary.iloc[(summary.sesgo_pct.abs()).argmin()]
engineer_row = summary.loc[summary.estrategia.eq("Ingeniero")].iloc[0]
print(f"Menor WAPE entre estrategias evaluadas: {best.estrategia} = {best.WAPE_pct:.2f}%.")
print(f"Sesgo más cercano a cero: {best_bias.estrategia} = {best_bias.sesgo_pct:+.2f}%.")
print(f"Ingeniero: WAPE {engineer_row.WAPE_pct:.2f}%, sesgo {engineer_row.sesgo_pct:+.2f}%.")
print("La corrección sólo es útil si mejora fuera del periodo de calibración; comparar también sesgo y error por H.")
print("Los tres resultados recientes disponibles no bastan para declarar ganador general ni desplegar el híbrido.")


Menor WAPE entre estrategias evaluadas: Ingeniero = 16.34%.
Sesgo más cercano a cero: Agrupado ajustado en noviembre = -5.48%.
Ingeniero: WAPE 16.34%, sesgo -7.51%.
La corrección sólo es útil si mejora fuera del periodo de calibración; comparar también sesgo y error por H.
Los tres resultados recientes disponibles no bastan para declarar ganador general ni desplegar el híbrido.
